# Web Scraping **fatsecret.co.id**

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

def scrape_nutrition_data(query_list):
    all_data = []
    base_url = "https://www.fatsecret.co.id"
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    for item in query_list:
        print(f"Memproses: {item}...")
        search_url = f"{base_url}/kalori-gizi/search?q={item}"

        row = {
            'nama makanan': item, 'kalori': '-', 'lemak': '-',
            'karbohidrat': '-', 'protein': '-', 'link': '-', 'status': '0'
        }

        try:
            # STEP 1: Pencarian
            res_search = requests.get(search_url, headers=headers)
            soup_search = BeautifulSoup(res_search.text, 'html.parser')

            # STEP 2: Cek No Result
            if not soup_search.find('div', class_='searchNoResult'):
                # STEP 3: Ambil link hasil pertama
                table = soup_search.find('table', class_='generic searchResult')
                if table:
                    first_tr = table.find('tr')
                    link_tag = first_tr.find('a', class_='prominent')

                    if link_tag:
                        row['nama makanan'] = link_tag.text.strip()
                        initial_url = base_url + link_tag['href']

                        # STEP 4: Akses URL Detail Awal
                        res_initial = requests.get(initial_url, headers=headers)
                        soup_initial = BeautifulSoup(res_initial.text, 'html.parser')

                        # --- STEP BARU: Cari link "100 gram" ---
                        # Mencari tag <a> yang teksnya mengandung "100 gram" atau "100 g"
                        link_100g = soup_initial.find('a', string=lambda s: s and "100 gram" in s.lower())

                        target_url = initial_url # default jika link 100g tidak ketemu
                        if link_100g and link_100g.has_attr('href'):
                            target_url = base_url + link_100g['href']
                            # Akses ulang ke halaman khusus porsi 100g
                            res_detail = requests.get(target_url, headers=headers)
                            soup_detail = BeautifulSoup(res_detail.text, 'html.parser')
                        else:
                            soup_detail = soup_initial

                        row['link'] = target_url

                        # STEP 5: Ambil nilai gizi
                        facts = soup_detail.find_all('td', class_='fact')
                        if facts:
                            for fact in facts:
                                title_el = fact.find('div', class_='factTitle')
                                value_el = fact.find('div', class_='factValue')

                                if title_el and value_el:
                                    title = title_el.text.strip().lower()
                                    value = value_el.text.strip()

                                    if 'kal' in title: row['kalori'] = value
                                    elif 'lemak' in title: row['lemak'] = value
                                    elif 'karb' in title: row['karbohidrat'] = value
                                    elif 'prot' in title: row['protein'] = value
                            row['status'] = '1'
            else:
                row['status'] = '0'

        except Exception as e:
            print(f"Error pada {item}: {e}")
            row['status'] = 'Error'

        all_data.append(row)
        time.sleep(1.5)

    # Simpan ke CSV
    df = pd.DataFrame(all_data)
    df.to_csv('hasil_gizi_100gram.csv', index=False, encoding='utf-8-sig')
    print("\nSelesai! Data diseragamkan ke porsi 100g (jika tersedia).")

# --- RUN ---
daftar_cari = ['AW cola', 'Beijing Beef', 'Fried Rice', 'Hashbrown', 'Honey Walnut Shrimp', 'Kung Pao Chicken', 'String Bean Chicken Breast', 'Super Greens', 'The Original Orange Chicken', 'White Steamed Rice', 'black pepper rice bowl', 'burger', 'carrot_eggs', 'cheese burger', 'chicken waffle', 'chicken_nuggets', 'chinese_cabbage', 'chinese_sausage', 'crispy corn', 'curry', 'french fries', 'fried chicken', 'fried_chicken', 'fried_dumplings', 'fried_eggs', 'mango chicken pocket', 'mozza burger', 'mung_bean_sprouts', 'nugget', 'perkedel', 'rice', 'sprite', 'tostitos cheese dip sauce', 'triangle_hash_brown', 'water_spinach']
scrape_nutrition_data(daftar_cari)

Memproses: AW cola...
Memproses: Beijing Beef...
Memproses: Fried Rice...
Memproses: Hashbrown...
Memproses: Honey Walnut Shrimp...
Memproses: Kung Pao Chicken...
Memproses: String Bean Chicken Breast...
Memproses: Super Greens...
Memproses: The Original Orange Chicken...
Memproses: White Steamed Rice...
Memproses: black pepper rice bowl...
Memproses: burger...
Memproses: carrot_eggs...
Memproses: cheese burger...
Memproses: chicken waffle...
Memproses: chicken_nuggets...
Memproses: chinese_cabbage...
Memproses: chinese_sausage...
Memproses: crispy corn...
Memproses: curry...
Memproses: french fries...
Memproses: fried chicken...
Memproses: fried_chicken...
Memproses: fried_dumplings...
Memproses: fried_eggs...
Memproses: mango chicken pocket...
Memproses: mozza burger...
Memproses: mung_bean_sprouts...
Memproses: nugget...
Memproses: perkedel...
Memproses: rice...
Memproses: sprite...
Memproses: tostitos cheese dip sauce...
Memproses: triangle_hash_brown...
Memproses: water_spinach...